# AASIST Fine-Tuning for Speech Deepfake Detection

This notebook fine-tunes the official AASIST model on locally available ASVspoof 2021 DF audio.

Methodological safeguards:

- the upstream AASIST source is pinned to a specific revision and is never rewritten;
- the official AASIST model configuration and class order are preserved;
- only audio files present on disk are included;
- train, validation, and test partitions are stratified and disjoint;
- model output is unpacked as `embedding, logits`, so the published two-class head is trained directly;
- evaluation reports balanced accuracy, macro F1, spoof F1, AUROC, EER, and a measured confusion matrix.

> ASVspoof 2021 DF is an evaluation corpus. Training on its released labels makes this a local research adaptation experiment, not an official challenge submission or benchmark result.


In [ ]:
# Install the notebook dependencies in Colab.
# For local environments, use: pip install -r requirements.txt
%pip install -q "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" \
    "librosa>=0.10.2,<1" "soundfile>=0.12,<1" "matplotlib>=3.8,<4" \
    "torch>=2.4,<3" "tqdm>=4.66,<5"


## 1. Runtime, reproducibility, and paths


In [ ]:
from __future__ import annotations

import json
import os
import random
import subprocess
import sys
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import soundfile as sf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


In [ ]:
DEFAULT_DATA_ROOT = (
    Path("/content/drive/MyDrive/PFEmaster")
    if IN_COLAB
    else Path("data")
)
DATA_ROOT = Path(os.getenv("AASIST_DATA_ROOT", DEFAULT_DATA_ROOT))
AUDIO_ROOT = Path(
    os.getenv(
        "ASVSPOOF_DF_AUDIO_ROOT",
        DATA_ROOT / "data" / "ASVspoof2021_DF_eval" / "flac",
    )
)
PROTOCOL_PATH = Path(
    os.getenv("ASVSPOOF_DF_PROTOCOL", DATA_ROOT / "trial_metadata.txt")
)
OUTPUT_ROOT = Path(
    os.getenv(
        "AASIST_OUTPUT_ROOT",
        "/content/aasist_finetuning_outputs" if IN_COLAB else "outputs",
    )
)
AASIST_ROOT = Path(
    os.getenv(
        "AASIST_SOURCE_ROOT",
        "/content/aasist" if IN_COLAB else "vendor/aasist",
    )
)

CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for directory in (CHECKPOINT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

AASIST_REPOSITORY = "https://github.com/clovaai/aasist.git"
AASIST_REVISION = "a04c9863f63d44471dde8a6abcb3b082b07cd1d1"

SAMPLE_RATE = 16_000
CLIP_SAMPLES = 64_600
BATCH_SIZE = 4
ACCUMULATION_STEPS = 4
EPOCHS = 10
LEARNING_RATE = 1e-5
TRAIN_SAMPLES_PER_EPOCH = 5_000
MAX_VALIDATION_SAMPLES = 5_000
VALIDATION_SIZE = 0.10
TEST_SIZE = 0.10
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

for label, path in {
    "Audio directory": AUDIO_ROOT,
    "Protocol": PROTOCOL_PATH,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

print("Audio:", AUDIO_ROOT)
print("Protocol:", PROTOCOL_PATH)
print("Outputs:", OUTPUT_ROOT)


In [ ]:
def run_git(*arguments: str, cwd: Path | None = None) -> None:
    subprocess.run(
        ["git", *arguments],
        cwd=cwd,
        check=True,
        text=True,
    )


if not (AASIST_ROOT / ".git").exists():
    AASIST_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run_git("clone", AASIST_REPOSITORY, str(AASIST_ROOT))

run_git("fetch", "--depth", "1", "origin", AASIST_REVISION, cwd=AASIST_ROOT)
run_git("checkout", "--detach", AASIST_REVISION, cwd=AASIST_ROOT)

resolved_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=AASIST_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if resolved_revision != AASIST_REVISION:
    raise RuntimeError(
        f"Expected AASIST revision {AASIST_REVISION}, got {resolved_revision}"
    )

if str(AASIST_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(AASIST_ROOT.resolve()))

with (AASIST_ROOT / "config" / "AASIST.conf").open(
    "r",
    encoding="utf-8",
) as handle:
    upstream_config = json.load(handle)

from models.AASIST import Model as AASISTModel

model = AASISTModel(upstream_config["model_config"])
pretrained_path = AASIST_ROOT / "models" / "weights" / "AASIST.pth"
pretrained_state = torch.load(
    pretrained_path,
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(pretrained_state, strict=True)
model = model.to(DEVICE)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Pinned AASIST revision:", resolved_revision)
print(f"Parameters: {parameter_count:,}")


## 2. Protocol parsing and matched audio index

The protocol describes more files than may be present in a partial local download. The pipeline builds an index from the actual audio directory and excludes missing files before any split is created.


In [ ]:
LABEL_TO_INDEX = {
    "spoof": 0,
    "bonafide": 1,
}
SPOOF_CLASS = LABEL_TO_INDEX["spoof"]
BONAFIDE_CLASS = LABEL_TO_INDEX["bonafide"]


def protocol_record(parts: list[str], track: str) -> tuple[str, str, str]:
    """Read released 2021 CM metadata or an explicit compact CM protocol.

    ASV target/nontarget trials are deliberately unsupported: nontarget is
    still bona-fide speech, not a spoof label.
    """
    track = track.upper()
    if track not in {"PA", "LA", "DF"}:
        raise ValueError(f"Unsupported track: {track}")
    if len(parts) >= 8 and parts[1].upper().startswith(f"{track}_E_"):
        label_position = 9 if track == "PA" else 5
        if len(parts) <= label_position + 2:
            raise ValueError("Incomplete ASVspoof 2021 CM metadata")
        file_id, label, subset = parts[1], parts[label_position], parts[label_position + 2]
    elif len(parts) in {2, 3}:
        file_id, label = parts[:2]
        subset = parts[2] if len(parts) == 3 else "unspecified"
    elif len(parts) == 5:
        # Conventional five-column CM protocol: speaker, trial, -, attack, key.
        file_id, label, subset = parts[1], parts[4], "unspecified"
    else:
        raise ValueError("Unsupported protocol schema; use CM metadata with bonafide/spoof keys")
    label = label.lower()
    file_id = Path(file_id).stem
    if label not in {"bonafide", "spoof"}:
        raise ValueError(f"Unknown CM label: {label!r}")
    if not file_id.upper().startswith(f"{track}_E_"):
        raise ValueError(f"Expected a {track} evaluation trial ID, got {file_id!r}")
    return file_id, label, subset


def load_protocol(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8-sig") as protocol:
        for line_number, raw_line in enumerate(protocol, start=1):
            if not raw_line.strip() or raw_line.lstrip().startswith("#"):
                continue
            try:
                rows.append(protocol_record(raw_line.split(), "DF"))
            except ValueError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error
    frame = pd.DataFrame(rows, columns=["file_id", "label", "usage_type"])
    if frame.empty:
        raise ValueError(f"Empty protocol: {path}")
    if (frame.groupby("file_id")["label"].nunique() > 1).any():
        raise ValueError("Conflicting labels for the same audio ID")
    return frame.drop_duplicates("file_id").reset_index(drop=True)


def build_audio_index(root: Path) -> dict[str, Path]:
    extensions = {".flac", ".wav", ".ogg"}
    result = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            if path.stem in result:
                raise ValueError(f"Ambiguous duplicate audio ID: {path.stem}")
            result[path.stem] = path
    return result


protocol_frame = load_protocol(PROTOCOL_PATH)
audio_index = build_audio_index(AUDIO_ROOT)
protocol_frame["path"] = protocol_frame["file_id"].map(audio_index)
matched_frame = (
    protocol_frame.dropna(subset=["path"])
    .drop_duplicates(subset=["file_id"])
    .reset_index(drop=True)
)
matched_frame["target"] = matched_frame["label"].map(LABEL_TO_INDEX).astype(int)

print(f"Protocol rows: {len(protocol_frame):,}")
print(f"Audio files indexed: {len(audio_index):,}")
print(f"Matched labeled files: {len(matched_frame):,}")
print(matched_frame.groupby(["usage_type", "label"]).size())

if matched_frame.empty:
    raise RuntimeError("No protocol entries matched audio files.")
if matched_frame["target"].nunique() != 2:
    raise RuntimeError("Both bona-fide and spoof classes are required.")


In [ ]:
def load_waveform(path: Path) -> np.ndarray:
    signal, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    signal = signal.mean(axis=1)
    if signal.size == 0 or not np.isfinite(signal).all():
        raise ValueError(f"Empty or non-finite audio: {path}")
    if sample_rate != SAMPLE_RATE:
        signal = librosa.resample(signal, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    peak = float(np.max(np.abs(signal)))
    if peak > 1.0:
        signal = signal / peak
    return signal.astype(np.float32)


def fit_clip(
    signal: np.ndarray,
    *,
    random_crop: bool,
) -> np.ndarray:
    if len(signal) == 0 or not np.isfinite(signal).all():
        raise ValueError("Cannot crop empty or non-finite audio")
    if len(signal) < CLIP_SAMPLES:
        repeats = int(np.ceil(CLIP_SAMPLES / max(len(signal), 1)))
        signal = np.tile(signal, repeats)

    if len(signal) == CLIP_SAMPLES:
        return signal.astype(np.float32)

    maximum_start = len(signal) - CLIP_SAMPLES
    if random_crop:
        start = random.randint(0, maximum_start)
    else:
        start = maximum_start // 2
    return signal[start : start + CLIP_SAMPLES].astype(np.float32)


class WaveformDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        *,
        training: bool,
    ):
        self.frame = frame.reset_index(drop=True)
        self.training = training

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, torch.Tensor, str]:
        row = self.frame.iloc[index]
        signal = fit_clip(
            load_waveform(Path(row.path)),
            random_crop=self.training,
        )

        if self.training and random.random() < 0.20:
            noise = np.random.normal(
                loc=0.0,
                scale=0.001,
                size=signal.shape,
            ).astype(np.float32)
            signal = np.clip(signal + noise, -1.0, 1.0)

        return (
            torch.from_numpy(signal),
            torch.tensor(int(row.target), dtype=torch.long),
            str(row.file_id),
        )


In [ ]:
train_frame, holdout_frame = train_test_split(
    matched_frame,
    test_size=VALIDATION_SIZE + TEST_SIZE,
    random_state=SEED,
    stratify=matched_frame["target"],
)
relative_test_size = TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)
validation_frame, test_frame = train_test_split(
    holdout_frame,
    test_size=relative_test_size,
    random_state=SEED,
    stratify=holdout_frame["target"],
)

train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

train_dataset = WaveformDataset(train_frame, training=True)
validation_dataset = WaveformDataset(validation_frame, training=False)
test_dataset = WaveformDataset(test_frame, training=False)

class_counts = train_frame["target"].value_counts().to_dict()
sample_weights = train_frame["target"].map(
    lambda label: 1.0 / class_counts[label]
).to_numpy()
sampler = WeightedRandomSampler(
    torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=min(TRAIN_SAMPLES_PER_EPOCH, len(train_dataset)),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
}
train_loader = DataLoader(
    train_dataset,
    sampler=sampler,
    drop_last=True,
    **loader_options,
)
if len(train_loader) == 0:
    raise ValueError("Training data is smaller than BATCH_SIZE; reduce BATCH_SIZE")
validation_loader = DataLoader(
    validation_dataset,
    shuffle=False,
    drop_last=False,
    **loader_options,
)
test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **loader_options,
)

for name, frame in {
    "train": train_frame,
    "validation": validation_frame,
    "test": test_frame,
}.items():
    frame.drop(columns="path").to_csv(OUTPUT_ROOT / f"{name}_manifest.csv", index=False)
    counts = frame["label"].value_counts().to_dict()
    print(name, len(frame), counts)


## 3. Fine-tuning

The official checkpoint uses class index 0 for spoof and index 1 for bona fide. That order is preserved so the pretrained classifier remains meaningful. Small-batch fine-tuning uses pretrained BatchNorm statistics, gradient accumulation, and automatic mixed precision on CUDA.


In [ ]:
def freeze_batch_normalization(module: nn.Module) -> None:
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        module.eval()


def equal_error_rate(
    labels: np.ndarray,
    spoof_probabilities: np.ndarray,
) -> float:
    """Empirical nearest-ROC-crossing EER estimate, not the challenge scorer."""
    if len(np.unique(labels)) != 2:
        raise ValueError("EER requires both classes")
    spoof_targets = (labels == SPOOF_CLASS).astype(np.int64)
    false_positive_rate, true_positive_rate, _ = roc_curve(
        spoof_targets,
        spoof_probabilities,
    )
    false_negative_rate = 1.0 - true_positive_rate
    index = int(
        np.nanargmin(np.abs(false_positive_rate - false_negative_rate))
    )
    return float(
        (false_positive_rate[index] + false_negative_rate[index]) / 2.0
    )


def metric_summary(
    labels: list[int],
    predictions: list[int],
    spoof_probabilities: list[float],
) -> dict[str, float]:
    if not labels or len(labels) != len(predictions) or len(labels) != len(spoof_probabilities):
        raise ValueError("Metrics require nonempty aligned labels, predictions and scores")
    if not np.isfinite(spoof_probabilities).all():
        raise ValueError("Non-finite prediction probabilities")
    label_array = np.asarray(labels)
    prediction_array = np.asarray(predictions)
    spoof_targets = (label_array == SPOOF_CLASS).astype(np.int64)

    result = {
        "accuracy": float(accuracy_score(label_array, prediction_array)),
        "balanced_accuracy": float(
            balanced_accuracy_score(label_array, prediction_array)
        ),
        "macro_f1": float(
            f1_score(
                label_array,
                prediction_array,
                average="macro",
                zero_division=0,
            )
        ),
        "spoof_f1": float(
            f1_score(
                label_array,
                prediction_array,
                pos_label=SPOOF_CLASS,
                average="binary",
                zero_division=0,
            )
        ),
    }
    if len(set(labels)) < 2:
        result.update(auroc=None, eer=None)
        return result
    try:
        result["auroc"] = float(
            roc_auc_score(spoof_targets, spoof_probabilities)
        )
        result["eer"] = equal_error_rate(
            label_array,
            np.asarray(spoof_probabilities),
        )
    except ValueError:
        result["auroc"] = None
        result["eer"] = None
    return result


@torch.no_grad()
def evaluate(
    loader: DataLoader,
    *,
    description: str,
    max_samples: int | None = None,
) -> tuple[dict[str, float], dict[str, list]]:
    model.eval()
    labels: list[int] = []
    predictions: list[int] = []
    spoof_probabilities: list[float] = []
    file_ids: list[str] = []

    for waveforms, batch_labels, batch_ids in tqdm(
        loader,
        desc=description,
        leave=False,
    ):
        waveforms = waveforms.to(DEVICE, non_blocking=True)
        _, logits = model(waveforms, Freq_aug=False)
        probabilities = torch.softmax(logits, dim=1)

        labels.extend(batch_labels.numpy().tolist())
        predictions.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        spoof_probabilities.extend(
            probabilities[:, SPOOF_CLASS].cpu().numpy().tolist()
        )
        file_ids.extend(batch_ids)

        if max_samples is not None and len(labels) >= max_samples:
            break

    details = {
        "file_id": file_ids[:max_samples],
        "label": labels[:max_samples],
        "prediction": predictions[:max_samples],
        "spoof_probability": spoof_probabilities[:max_samples],
    }
    metrics = metric_summary(
        details["label"],
        details["prediction"],
        details["spoof_probability"],
    )
    return metrics, details


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)
gradient_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=DEVICE.type == "cuda",
)


def train_epoch() -> dict[str, float]:
    model.train()
    model.apply(freeze_batch_normalization)
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    correct = 0
    samples = 0

    for step, (waveforms, labels, _) in enumerate(
        tqdm(train_loader, desc="Training", leave=False),
        start=1,
    ):
        waveforms = waveforms.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=DEVICE.type == "cuda",
        ):
            _, logits = model(waveforms, Freq_aug=False)
            loss = criterion(logits, labels)
            window_start = ((step - 1) // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
            window_size = min(ACCUMULATION_STEPS, len(train_loader) - window_start)
            scaled_loss = loss / window_size

        gradient_scaler.scale(scaled_loss).backward()

        should_step = (
            step % ACCUMULATION_STEPS == 0
            or step == len(train_loader)
        )
        if should_step:
            gradient_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            gradient_scaler.step(optimizer)
            gradient_scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        samples += labels.size(0)

    return {
        "loss": total_loss / max(samples, 1),
        "accuracy": correct / max(samples, 1),
    }


def save_checkpoint(
    path: Path,
    *,
    epoch: int,
    history: list[dict],
) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "gradient_scaler_state_dict": gradient_scaler.state_dict(),
            "history": history,
            "upstream_revision": AASIST_REVISION,
            "label_to_index": LABEL_TO_INDEX,
        },
        path,
    )


In [ ]:
history: list[dict] = []
best_macro_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_epoch()
    validation_metrics, _ = evaluate(
        validation_loader,
        description=f"Validation epoch {epoch}",
        max_samples=MAX_VALIDATION_SAMPLES,
    )
    record = {
        "epoch": epoch,
        "train": train_metrics,
        "validation": validation_metrics,
    }
    history.append(record)
    print(json.dumps(record, indent=2))

    save_checkpoint(
        CHECKPOINT_ROOT / "last.pt",
        epoch=epoch,
        history=history,
    )
    if validation_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = validation_metrics["macro_f1"]
        save_checkpoint(
            CHECKPOINT_ROOT / "best.pt",
            epoch=epoch,
            history=history,
        )

with (OUTPUT_ROOT / "training_history.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(history, handle, indent=2, allow_nan=False)

print("Best validation macro F1:", best_macro_f1)


## 4. Held-out test evaluation

The test partition is evaluated once after model selection. Every reported value below is calculated from model predictions; no illustrative or manually constructed confusion matrix is used.


In [ ]:
best_checkpoint = torch.load(
    CHECKPOINT_ROOT / "best.pt",
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)

test_metrics, test_details = evaluate(
    test_loader,
    description="Held-out test",
)
print(json.dumps(test_metrics, indent=2))
print(
    classification_report(
        test_details["label"],
        test_details["prediction"],
        labels=[BONAFIDE_CLASS, SPOOF_CLASS],
        target_names=["bonafide", "spoof"],
        digits=4,
        zero_division=0,
    )
)

matrix = confusion_matrix(
    test_details["label"],
    test_details["prediction"],
    labels=[BONAFIDE_CLASS, SPOOF_CLASS],
)
figure, axis = plt.subplots(figsize=(5, 4), dpi=140)
image = axis.imshow(matrix, cmap="Blues")
axis.set(
    title="Held-out ASVspoof DF test split",
    xlabel="Predicted",
    ylabel="True",
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["bonafide", "spoof"],
    yticklabels=["bonafide", "spoof"],
)
for (row, column), value in np.ndenumerate(matrix):
    axis.text(column, row, str(value), ha="center", va="center")
figure.colorbar(image, ax=axis)
figure.tight_layout()
figure.savefig(
    FIGURE_ROOT / "test_confusion_matrix.png",
    bbox_inches="tight",
)
plt.show()

pd.DataFrame(test_details).to_csv(
    OUTPUT_ROOT / "test_predictions.csv",
    index=False,
)
with (OUTPUT_ROOT / "test_metrics.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(test_metrics, handle, indent=2, allow_nan=False)


In [ ]:
@torch.no_grad()
def predict_audio(path: str | Path) -> dict[str, float | str]:
    model.eval()
    signal = fit_clip(
        load_waveform(Path(path)),
        random_crop=False,
    )
    waveform = torch.from_numpy(signal).unsqueeze(0).to(DEVICE)
    _, logits = model(waveform, Freq_aug=False)
    probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

    predicted_index = int(np.argmax(probabilities))
    index_to_label = {
        index: label
        for label, index in LABEL_TO_INDEX.items()
    }
    return {
        "prediction": index_to_label[predicted_index],
        "spoof_probability": float(probabilities[SPOOF_CLASS]),
        "bonafide_probability": float(probabilities[BONAFIDE_CLASS]),
    }


# Example:
# predict_audio(AUDIO_ROOT / "DF_E_2000011.flac")


## 5. Interpretation and limits

- Accuracy is not sufficient for this class-imbalanced corpus. Use balanced accuracy, macro F1, per-class results, AUROC, and EER together.
- A model that predicts every file as spoof can achieve high raw accuracy while completely failing to recognize bona-fide speech.
- The local random split is a development convenience. It does not replace official ASVspoof protocols, cross-corpus evaluation, or attack-condition analysis.
- This repository does not publish the original notebook's saved metrics because its test run predicted every sample as spoof and its adapter trained on the 640-dimensional hidden representation instead of the official two-logit output.
